# 21-тарау. Мысық па, ит пе? Трансферлік оқыту

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 21.1. Жобаның мақсаты

## 21.2. Трансферлік оқыту қалай жұмыс істейді?

## 21.3. 1-қадам. Деректерді жүктеу

In [ ]:
URL = ("https://storage.googleapis.com/mledu-datasets/"
       "cats_and_dogs_filtered.zip")
!wget -q {URL}
!unzip -q cats_and_dogs_filtered.zip

In [ ]:
import numpy as np
import keras
from keras import layers

IMG_SIZE = (160, 160)
train_ds = keras.utils.image_dataset_from_directory(
    "cats_and_dogs_filtered/train", image_size=IMG_SIZE,
    batch_size=32, shuffle=True, seed=42)
val_ds = keras.utils.image_dataset_from_directory(
    "cats_and_dogs_filtered/validation", image_size=IMG_SIZE,
    batch_size=32, shuffle=False)
print(train_ds.class_names)

## 21.4. 2-қадам. Деректерді кеңейту

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),      # айналы бейнелеу
    layers.RandomRotation(0.1),           # ±10% айналымға бұру
    layers.RandomZoom(0.1),               # ±10% үлкейту/кішірейту
])

## 21.5. 3-қадам. Модельді құрастыру

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,           # ImageNet-тің 1000 класты қабатынсыз
    weights="imagenet")          # алдын ала оқытылған салмақтар
base.trainable = False           # мұздату

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)  # → [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)    # әр белгі картасының орташасы
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)   # ит болу ықтималдығы
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])

## 21.6. 4-қадам. Оқыту

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

## 21.7. 5-қадам. Fine-tuning: нәтижені жетілдіру

In [ ]:
base.trainable = True
for layer in base.layers[:100]:   # алғашқы 100 қабат мұздатулы қалады
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),   # 100 есе кіші қарқын!
              loss="binary_crossentropy", metrics=["accuracy"])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 21.8. 6-қадам. Өз суретіңізді тексеру

**Өз суретіңіз.** Үй жануарыңыздың суретін `my_pet.jpg` атымен Colab-қа жүктеңіз.

In [ ]:
img = keras.utils.load_img("my_pet.jpg", target_size=IMG_SIZE)
x = keras.utils.img_to_array(img)[np.newaxis]     # (1, 160, 160, 3)
p = float(model.predict(x, verbose=0)[0, 0])
if p > 0.5:
    print(f"Бұл — ит ({p:.0%})")
else:
    print(f"Бұл — мысық ({1 - p:.0%})")

model.save("cats_dogs.keras")